# GR00T N1.5 resident adapter (`--backend groot-n15`)

Serves GR00T N1.5 through NVIDIA Isaac-GR00T's own `Gr00tPolicy` from a resident Flask adapter, `control/vla_bridge/groot_n15_server.py`, through an ngrok HTTP tunnel.

| | |
|---|---|
| Lane | HTTP adapter (isolated Python 3.10 venv) |
| Port → tunnel | 3006 → ngrok HTTP |
| GPU runtime | L4 |
| Gated HF licences | none |
| Sign-offs before running | none |
| Default checkpoint | `quangnd58/GR00T-N1.5-3B-so101-multitask@488f3b6` |
| Server log | `/content/groot_n15_server.log` |
| Embedded files (written to `/content`) | `colab_server_utils.py`, `groot_n15_server.py` |

Run top to bottom on a fresh runtime. Works from the VS Code Colab extension: nothing to upload.
Every file this notebook runs is written into `/content` by a cell below and checked against
its sha256. Run the teardown cell at the end of every session. The index of all notebooks,
with the local smoke and episode commands, is `control/vla_bridge/notebooks/README.md`.

Generated by `scripts/build_colab_notebooks.py` from `scripts/colab_notebook_specs.py` +
`control/vla_bridge` sources, so edit those and regenerate, not this file.

In [ ]:
import os

# Test 7 UAT crash investigation (Phase 12, 2026-09-27): Colab sessions that
# load multiple ML libraries (torch, opencv-headless, Fast-FoundationStereo's
# own deps) commonly end up with more than one bundled OpenMP runtime
# (libgomp vs libiomp) loaded into the same process -- the classic
# "OMP: Error #15" failure mode, which manifests as an abrupt, tracebackless
# kernel death the first time a *new* library's compute path runs for the
# first time (matches "crashes right as inference starts"). Set before any
# heavy imports so it applies process-wide to every cell below.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
print("OpenMP duplicate-runtime guards set.")

**pyngrok legitimacy gate:** the next cell installs `pyngrok`, flagged
`SUS` in `11-RESEARCH.md`'s Package Legitimacy Audit. Do **not** run that
install cell until a human has completed the verification in this plan's
Task 3 `checkpoint:human-verify` (visit pypi.org/project/pyngrok, confirm
the `ngrok`/`alexdlaird` maintainer namespace and that `8.1.2` is a real
published version).

In [ ]:
# Colab notebook cell — DO NOT RUN until the human has completed the
# pyngrok legitimacy check in this plan's Task 3 checkpoint:human-verify
!pip install pyngrok==8.1.2

In [ ]:
# One-time token setup: HF Hub (for authenticated, faster/rate-limit-free checkpoint
# downloads) + ngrok (for the TCP tunnel). Checks for saved token files on Google Drive
# first (same MyDrive/SoARM-Research/.hf_token / .ngrok_token convention as other
# cells in this project), else falls back to a one-time getpass() prompt so tokens
# are never echoed or left in cell output/notebook history.
import getpass
from pathlib import Path

_hf_token = None
_ngrok_token = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _hf_token_path = Path("/content/drive/MyDrive/SoARM-Research/.hf_token")
    _ngrok_token_path = Path("/content/drive/MyDrive/SoARM-Research/.ngrok_token")
    if _hf_token_path.exists():
        _hf_token = _hf_token_path.read_text().strip()
        print(f"HF token loaded from {_hf_token_path}")
    if _ngrok_token_path.exists():
        _ngrok_token = _ngrok_token_path.read_text().strip()
        print(f"ngrok token loaded from {_ngrok_token_path}")
except Exception as _exc:
    print(f"Drive mount/token read skipped ({type(_exc).__name__}: {_exc})")

if not _hf_token:
    _hf_token = getpass.getpass("HF Hub token (paste here): ")
if not _ngrok_token:
    _ngrok_token = getpass.getpass("ngrok authtoken (paste here): ")

from huggingface_hub import login
login(token=_hf_token)
print("Logged into HF Hub.")

from pyngrok import ngrok
ngrok.set_auth_token(_ngrok_token)
print("ngrok authtoken set.")

del _hf_token, _ngrok_token

## Server helper

The next cell writes `control/vla_bridge/colab_server_utils.py` into `/content/colab_server_utils.py`; the cell after it checks its sha256. The launch, diagnostic and teardown cells import it to stop stale servers, prove this notebook's server owns its port, and report logs, processes and GPU memory.

In [ ]:
%%writefile /content/colab_server_utils.py
"""Colab-side process/port helper for the per-backend policy-server notebooks.

Every notebook in control/vla_bridge/notebooks/ writes this file to
/content/colab_server_utils.py with a %%writefile cell (byte-identical to this file, checked
by a sha256 cell and by control/test_colab_notebooks.py) and imports it with
sys.path.insert(0, '/content'). Local control code never imports it.

What it guarantees for a launch cell:
- stop_servers() clears any earlier server: every process whose cmdline contains a known
  server marker, plus whatever is listening on the notebook's port.
- start_server() starts the server unbuffered (PYTHONUNBUFFERED=1) with its log in a file.
- wait_until_ready() only succeeds when the process this cell started (or one of its
  children) owns the port. A stale server that answers on the port is an error, not a
  success. It also fails on an early exit and, for patched launchers, on a missing banner.

The diagnostic helpers (log_report, process_report, gpu_report, health_report) print what a
partner needs to debug a run, and stop_server() is the teardown.

Imports: stdlib + psutil only (psutil is preinstalled on Colab).
"""

import json
import os
import re
import socket
import subprocess
import time
import urllib.request
from pathlib import Path
from typing import Callable, Mapping, Optional, Sequence

import psutil

SERVER_MARKERS = (
    "lerobot.async_inference.policy_server",
    "diffusion_policy_server.py",
    "groot_n17_server.py",
    "fastfs_server.py",
    "openvla_oft_server.py",
    "openvla_server.py",
    "octo_server.py",
    "wallx_server.py",
    "rdt1b_server.py",
    "groot_n15_server.py",
)

# Case-sensitive on purpose: "Error" covers KeyError, NotImplementedError, GatedRepoError and
# "Error in StreamActions"; lowercase prose such as "error budget" is not a match.
ERROR_PATTERN = re.compile(
    r"ERROR|Error|Traceback|Exception|OutOfMemory|out of memory|Killed|beginAllocateToPool"
)

_ACCESS_ERRORS = (psutil.AccessDenied, psutil.NoSuchProcess, psutil.ZombieProcess)


def _cmdline(proc) -> str:
    try:
        return " ".join(proc.cmdline())
    except _ACCESS_ERRORS:
        return ""
    except OSError:
        return ""


def _describe(pid: int) -> str:
    try:
        return f"PID {pid} ({_cmdline(psutil.Process(pid)) or '?'})"
    except _ACCESS_ERRORS:
        return f"PID {pid} (gone)"


def _tcp_connections(proc):
    # Colab's psutil may predate 6.0, where net_connections() replaced connections().
    method = getattr(proc, "net_connections", None) or proc.connections
    return method(kind="tcp")


def _read_lines(log_path) -> list:
    path = Path(log_path)
    if not path.exists():
        return []
    return path.read_text(errors="replace").splitlines()


def _tail(log_path, lines: int = 40) -> str:
    return "\n".join(_read_lines(log_path)[-lines:])


def listening_pids(port: int) -> set:
    """PIDs of every visible process with a TCP socket in LISTEN state on `port`."""
    pids = set()
    for proc in psutil.process_iter():
        try:
            for conn in _tcp_connections(proc):
                if conn.status == psutil.CONN_LISTEN and conn.laddr and conn.laddr.port == port:
                    pids.add(proc.pid)
        except _ACCESS_ERRORS:
            continue
        except OSError:
            continue
    return pids


def stop_servers(port: int, patterns: Sequence[str] = (), timeout_s: float = 15.0) -> list:
    """Stop every process matching a pattern plus the port's listener; return what was stopped."""
    me = os.getpid()
    targets = {}
    for proc in psutil.process_iter():
        if proc.pid == me:
            continue
        cmdline = _cmdline(proc)
        if cmdline and any(pattern and pattern in cmdline for pattern in patterns):
            targets[proc.pid] = proc
    for pid in listening_pids(port):
        if pid != me and pid not in targets:
            try:
                targets[pid] = psutil.Process(pid)
            except _ACCESS_ERRORS:
                continue
    stopped = []
    for pid, proc in sorted(targets.items()):
        description = f"PID {pid} ({_cmdline(proc) or '?'})"
        try:
            proc.terminate()
            stopped.append(description)
        except psutil.NoSuchProcess:
            continue
        except psutil.AccessDenied:
            print(f"Cannot stop {description}: access denied")
    _, alive = psutil.wait_procs(list(targets.values()), timeout=5)
    for proc in alive:
        try:
            proc.kill()
        except psutil.NoSuchProcess:
            pass
    psutil.wait_procs(alive, timeout=5)
    for line in stopped:
        print(f"Stopped {line}")
    deadline = time.monotonic() + timeout_s
    while True:
        holders = listening_pids(port) - {me}
        if not holders:
            return stopped
        if time.monotonic() >= deadline:
            raise RuntimeError(
                f"port {port} is still held by "
                + ", ".join(_describe(pid) for pid in sorted(holders))
                + f" after {timeout_s:.0f} s; stop it by hand before launching"
            )
        time.sleep(0.2)


def start_server(argv: Sequence[str], log_path,
                 env_overrides: Optional[Mapping[str, str]] = None,
                 cwd=None) -> subprocess.Popen:
    """Start `argv` unbuffered with stdout+stderr in `log_path` (truncated)."""
    env = os.environ.copy()
    env.update(env_overrides or {})
    env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = open(log_path, "wb", buffering=0)
    try:
        proc = subprocess.Popen(list(argv), stdout=log, stderr=subprocess.STDOUT, env=env,
                                cwd=None if cwd is None else str(cwd))
    finally:
        log.close()
    print(f"Started PID {proc.pid}: {' '.join(argv)}")
    print(f"Log → {log_path}")
    return proc


def _probe(port: int, health_url: Optional[str], ready: Optional[Callable[[dict], bool]]):
    """Return (answered, health): answered is True once the server counts as ready."""
    if health_url:
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                health = json.loads(response.read().decode())
        except Exception:
            return False, None
        if not isinstance(health, dict):
            return False, None
        ok = ready(health) if ready is not None else health.get("status") == "ready"
        return bool(ok), health
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True, None
    except OSError:
        return False, None


def _own_pids(proc) -> set:
    pids = {proc.pid}
    try:
        pids.update(child.pid for child in psutil.Process(proc.pid).children(recursive=True))
    except _ACCESS_ERRORS:
        pass
    return pids


def wait_until_ready(proc, port: int, log_path, timeout_s: float,
                     health_url: Optional[str] = None,
                     ready: Optional[Callable[[dict], bool]] = None,
                     banner: Optional[str] = None,
                     settle_s: float = 2.0, poll_s: float = 1.0):
    """Wait until the server this cell started owns `port` and is ready.

    Returns the /health dict in health mode, None in TCP mode. Raises RuntimeError on an
    early exit, a foreign/stale owner or a missing launcher banner, and TimeoutError (after
    terminating `proc`) when the deadline passes.
    """
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        code = proc.poll()
        if code is not None:
            raise RuntimeError(
                f"server PID {proc.pid} exited with code {code} before it was ready. "
                f"Last log lines of {log_path}:\n{_tail(log_path)}"
            )
        answered, health = _probe(port, health_url, ready)
        if answered:
            owners = listening_pids(port)
            mine = _own_pids(proc)
            if not owners or not owners <= mine:
                foreign = ", ".join(_describe(pid) for pid in sorted(owners - mine))
                foreign = foreign or "no visible process"
                raise RuntimeError(
                    f"port {port} answered but is owned by {foreign}, not the server this cell "
                    f"started (PID {proc.pid}). That is a stale server; re-run the launch cell, "
                    "which stops old servers first."
                )
            time.sleep(settle_s)
            code = proc.poll()
            if code is not None:
                raise RuntimeError(
                    f"server PID {proc.pid} exited with code {code} right after binding port "
                    f"{port}. Last log lines of {log_path}:\n{_tail(log_path)}"
                )
            if banner is not None:
                text = Path(log_path).read_text(errors="replace") if Path(log_path).exists() else ""
                if banner not in text:
                    raise RuntimeError(
                        f"expected launcher banner '{banner}' not found in {log_path}, so this "
                        "is not the patched launcher. Re-run the embed cells, then the launch cell."
                    )
            print(f"READY: PID {proc.pid} ({_cmdline(psutil.Process(proc.pid))}) owns port "
                  f"{port}; log → {log_path}")
            return health
        time.sleep(poll_s)
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
    raise TimeoutError(
        f"server PID {proc.pid} was not ready on port {port} within {timeout_s:.0f} s "
        f"(terminated). Last log lines of {log_path}:\n{_tail(log_path)}"
    )


def log_report(log_path, tail: int = 80, max_matches: int = 40) -> None:
    """Print the last `tail` log lines and up to `max_matches` numbered error lines."""
    path = Path(log_path)
    if not path.exists():
        print(f"{path} does not exist (has the launch cell run on this runtime?)")
        return
    lines = _read_lines(path)
    print(f"--- last {tail} lines of {path} ({len(lines)} lines total) ---")
    for line in lines[-tail:]:
        print(line)
    matches = [(number, line) for number, line in enumerate(lines, start=1)
               if ERROR_PATTERN.search(line)]
    print(f"--- {len(matches)} lines matching ERROR_PATTERN (showing up to {max_matches}) ---")
    for number, line in matches[:max_matches]:
        print(f"{number}: {line}")
    if not matches:
        print("(none)")


def process_report(port: int, expected_marker: str, proc=None) -> None:
    """Print every known server process, then check the right server owns `port`."""
    print("--- running server processes ---")
    found = False
    now = time.time()
    for candidate in psutil.process_iter():
        cmdline = _cmdline(candidate)
        if not cmdline or not any(marker in cmdline for marker in SERVER_MARKERS):
            continue
        try:
            elapsed = now - candidate.create_time()
            rss_mb = candidate.memory_info().rss / 1e6
        except _ACCESS_ERRORS:
            continue
        found = True
        print(f"PID {candidate.pid}  up {elapsed:7.0f} s  RSS {rss_mb:8.1f} MB  {cmdline}")
    if not found:
        print("(no process with a known server marker)")
    owners = listening_pids(port)
    if not owners:
        raise RuntimeError(f"nothing is listening on port {port}; run the launch cell")
    for pid in sorted(owners):
        try:
            cmdline = _cmdline(psutil.Process(pid))
        except _ACCESS_ERRORS:
            cmdline = ""
        if expected_marker not in cmdline:
            raise RuntimeError(
                f"WRONG server on port {port}: PID {pid} ({cmdline or '?'}) does not contain "
                f"'{expected_marker}'. Run the teardown cell, then the launch cell."
            )
    if proc is not None:
        mine = _own_pids(proc)
        if not owners <= mine:
            raise RuntimeError(
                f"port {port} is owned by "
                + ", ".join(_describe(pid) for pid in sorted(owners))
                + f", not by the server this notebook started (PID {proc.pid})"
            )
    print(f"OK: {expected_marker} (PID {min(owners)}) owns port {port}")


def gpu_report() -> None:
    """Print GPU memory/utilisation and the processes using the GPU."""
    queries = (
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv"],
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
    )
    for argv in queries:
        try:
            result = subprocess.run(argv, capture_output=True, text=True, timeout=30)
        except FileNotFoundError:
            print("no nvidia-smi (CPU runtime?): pick a GPU runtime for this notebook")
            return
        print(result.stdout.rstrip() or result.stderr.rstrip())


def health_report(port: int) -> dict:
    """GET /health on the local server, pretty-print it and return it."""
    url = f"http://127.0.0.1:{port}/health"
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            health = json.loads(response.read().decode())
    except Exception as exc:
        print(f"GET {url} failed: {type(exc).__name__}: {exc}")
        raise
    print(json.dumps(health, indent=2, sort_keys=True))
    return health


def stop_server(proc, port: int, patterns: Sequence[str] = ()) -> None:
    """Teardown: stop `proc` (if alive) and any leftover server, then confirm the port is free."""
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
        print(f"Stopped PID {proc.pid}")
    stop_servers(port, patterns)
    print(f"port {port} is free")


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '2d8f63cbfdf5e679c268dd4f4436129e8fa8f30acfd5c5f5d88eb1082c2bf2a8'
actual = hashlib.sha256(Path('/content/colab_server_utils.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/colab_server_utils.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/colab_server_utils.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/colab_server_utils.py matches control/vla_bridge/colab_server_utils.py (sha256 {actual[:12]})')

## Tier C: GR00T N1.5 resident adapter (Plan 14-12, `--backend groot-n15`)

GR00T N1.5 has no LeRobot path in this repo's lerobot 0.6.1: its `groot` policy is N1.7-only and rejects
N1.5 checkpoints (Plan 14-07). N1.5 therefore runs as its own resident Flask adapter,
`control/vla_bridge/groot_n15_server.py`, built like the RDT-1B and Octo adapters (see README): it loads once,
warms up, then serves. It does not use the gRPC PolicyServer lane, and it is a **separate code path from
GR00T N1.7 (`grpc_groot_n17.ipynb`)** (14-RESEARCH Pitfall 3).

The code comes from **github.com/NVIDIA/Isaac-GR00T**, cloned at the tag `n1.5-release`
(= branch `n1d5`, commit `4af2b62`, Apache-2.0). It is not on PyPI. The adapter loads checkpoints with the
repo's own `gr00t.model.policy.Gr00tPolicy` and the `so100_dualcam` data config, exactly as NVIDIA's SO-101
tutorial (huggingface.co/blog/nvidia/gr00t-n1-5-so101-tuning) and `examples/SO-100/eval_lerobot.py` do.

**Status:** written from upstream source and HF Hub metadata. These cells have **not yet been run
on Colab**.

Requirements:
- **GPU: an L4 or A100 runtime.** The bundled Eagle 2.5 backbone hard-codes `flash_attention_2`, which
  needs an Ampere-or-newer GPU. A T4 cannot run it, so `groot_n15_server.py` refuses to start there. The
  model card measures about 6 GB of VRAM for PyTorch inference.
- **Disk: about 10 GB free** (7.6 GB of checkpoint safetensors; optimizer state is not downloaded).
- **Dependencies:** Isaac-GR00T pins Python 3.10, `torch==2.5.1`, `transformers==4.51.3` and
  `flash-attn==2.7.1.post4`, so the adapter runs in its own venv. `groot_n15_server.py` refuses other
  versions, and refuses an Isaac-GR00T checkout at any other commit.

Default checkpoint: `quangnd58/GR00T-N1.5-3B-so101-multitask@488f3b6`, a documented SO-101 fine-tune of
`nvidia/GR00T-N1.5-3B` (Isaac-GR00T `n1d5`, `new_embodiment`, `so100_dualcam`: `video.front` = top camera
+ `video.wrist`). The base `nvidia/GR00T-N1.5-3B` cannot drive SO-101: SO-101 is not in its pre-training,
and its metadata has no `new_embodiment` entry. The action is an **absolute joint target in lerobot
degree mode**, so the client passes it straight through. **No IK, no delta.** The adapter re-derives the
unit mode from the checkpoint's own `metadata.json` and refuses to start if it is not degrees.

In [ ]:
# GR00T N1.5 only: the official install (README / NVIDIA's SO-101 tutorial: python 3.10,
# `pip install -e .[base]`, then flash-attn 2.7.1.post4 without build isolation) in an isolated venv,
# at the pinned NVIDIA/Isaac-GR00T n1.5-release commit. flash-attn's setup fetches its prebuilt
# wheel from github.com/Dao-AILab/flash-attention releases (or compiles, which takes a long time).
# Clone, installs and import check are all verified by exit status.
import subprocess, sys
from pathlib import Path
GROOT_N15_COMMIT = '4af2b622892f7dcb5aae5a3fb70bcb02dc217b96'  # tag n1.5-release == branch n1d5
GROOT_N15_DIR = '/content/Isaac-GR00T'
GROOT_N15_VENV = '/content/groot_n15_venv'
GROOT_N15_PYTHON = f'{GROOT_N15_VENV}/bin/python'
if not Path(GROOT_N15_DIR).exists():
    subprocess.run(['git', 'clone', 'https://github.com/NVIDIA/Isaac-GR00T', GROOT_N15_DIR], check=True)
subprocess.run(['git', '-C', GROOT_N15_DIR, 'checkout', '--detach', GROOT_N15_COMMIT], check=True)
head = subprocess.run(['git', '-C', GROOT_N15_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True)
if head.stdout.strip() != GROOT_N15_COMMIT:
    raise RuntimeError(f'Isaac-GR00T checkout is at {head.stdout.strip()}, expected {GROOT_N15_COMMIT}')
if not Path(GROOT_N15_PYTHON).exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
    subprocess.run(['uv', 'venv', '--python', '3.10', '--seed', GROOT_N15_VENV], check=True)
for packages in (
    ['--upgrade', 'setuptools'],
    ['-e', f'{GROOT_N15_DIR}[base]'],
    ['packaging', 'psutil', 'ninja', 'wheel', 'flask'],
    ['--no-build-isolation', 'flash-attn==2.7.1.post4'],
):
    install = subprocess.run(['uv', 'pip', 'install', '--python', GROOT_N15_PYTHON, *packages],
                             capture_output=True, text=True)
    print(install.stdout[-2000:]); print(install.stderr[-2000:])
    if install.returncode != 0:
        raise RuntimeError(f'GR00T N1.5 venv install failed (exit {install.returncode}): {packages}')
check = subprocess.run([GROOT_N15_PYTHON, '-c', (
    "import importlib.metadata as m, torch\n"
    "print({p: m.version(p) for p in ('torch', 'transformers', 'flash_attn', 'numpy', 'huggingface_hub')})\n"
    "import flash_attn, gr00t\n"
    "from gr00t.model.policy import Gr00tPolicy\n"
    "from gr00t.experiment.data_config import DATA_CONFIG_MAP\n"
    "assert 'so100_dualcam' in DATA_CONFIG_MAP\n"
    "print('gr00t from', gr00t.__file__)\n"
    "assert torch.cuda.is_available(), 'no CUDA GPU'\n"
    "cap = torch.cuda.get_device_capability(0)\n"
    "print(torch.cuda.get_device_name(0), cap, round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')\n"
    "assert cap >= (8, 0), 'flash-attn 2 needs Ampere or newer (T4 is sm75): switch the runtime to an L4/A100'\n"
)], cwd='/content', capture_output=True, text=True)
print(check.stdout); print(check.stderr[-2000:])
if check.returncode != 0:
    raise RuntimeError('GR00T N1.5 venv import check failed; see stderr above')

## Adapter source and launch notes

The next cell writes `control/vla_bridge/groot_n15_server.py` into `/content/groot_n15_server.py`, and the cell after it checks its sha256. The launch cell then starts it as a separate OS process.

It runs with the venv's Python.

On first start it downloads the checkpoint at its pinned revision (`config.json`, `experiment_cfg/`, the
safetensors and their index: about 7.6 GB). Before declaring the model ready, it:
- checks the clone is at the reviewed commit, and that `gr00t` is imported from it;
- checks the checkpoint's `metadata.json` is the reviewed layout (`video.front` + `video.wrist`,
  `single_arm` [5] + `gripper` [1], both `absolute`) and that its action statistics are **degree mode**;
- loads `Gr00tPolicy(..., embodiment_tag='new_embodiment', so100_dualcam, denoising_steps=4)` and checks
  every weight key against the checkpoint's index;
- runs one warm-up inference.

`/health` only answers after that. It reports `action_space` (`absolute_joint`), `action_units`
(`degrees`), `detected_action_units`, `chunk_step` and `video_sources`. The log at
`/content/groot_n15_server.log` records the GPU and the **peak VRAM** after warm-up.

The server returns the chunk step about one client control period ahead. By default that is
`--client-control-hz 2.0` (step 14 of 16), matching `run_vla_episode.py`'s default `--control-hz`. If you
run the client at another rate, pass the same value in the launch cell's command. `video.front` is the
dataset's top camera; it is fed from this rig's `overhead_left` (`--front-image-key overhead_right` to
change it).

In [ ]:
%%writefile /content/groot_n15_server.py
"""Resident GR00T N1.5 HTTP adapter (Tier C, Plan 14-12, VLAB-11). Run on Colab inside
the isolated Isaac-GR00T venv (see vla_bridge/notebooks/http_groot_n15.ipynb).

Same shape as `rdt1b_server.py` / `octo_server.py` / `fastfs_server.py`:
- the model loads ONCE
- one throwaway warmup inference runs before the socket opens
- `/health` reports ready
- a non-blocking `inference_lock` answers concurrent requests with 429
- `ValueError`/`KeyError`/`TypeError` on a request become 400
- a model failure (`RuntimeError`, e.g. a non-finite or wrong-shaped action)
  becomes 500, so the client holds position instead of parsing NaN JSON

The server stays model-format-only: it returns one step of the checkpoint's
un-normalized action chunk as 6 floats in `JOINT_ORDER`, and
`backends/groot_n15.py` reshapes them into the `JOINT_ORDER` dict.

Why this is NOT the GR00T N1.7 code path (14-RESEARCH Pitfall 3)
-----------------------------------------------------------------
lerobot 0.6.1's `groot` policy wraps N1.7 only and rejects N1.5 checkpoints
("GR00T N1.5 support was removed from LeRobot", Plan 14-07). N1.5 is served
through NVIDIA's own `NVIDIA/Isaac-GR00T` package, GitHub-only (not on PyPI;
14-RESEARCH audit), cloned at the tag `n1.5-release` = branch `n1d5` =
`ISAAC_GROOT_COMMIT`. Nothing here imports lerobot.

Verified against live upstream source (2026-10-01, execution discipline)
------------------------------------------------------------------------
github.com/NVIDIA/Isaac-GR00T @ 4af2b62 (Apache-2.0). Files read: `README.md`,
`pyproject.toml`, `gr00t/model/policy.py`, `gr00t/model/gr00t_n1.py`,
`gr00t/model/transforms.py`, `gr00t/model/backbone/eagle_backbone.py`,
`gr00t/model/backbone/eagle2_hg_model/config.json`,
`gr00t/experiment/data_config.py`, `gr00t/data/embodiment_tags.py`,
`gr00t/data/transform/{base,video,state_action,concat}.py`,
`gr00t/eval/service.py`, `scripts/inference_service.py`,
`examples/SO-100/{README.md,eval_lerobot.py,so100_dualcam__modality.json}`,
`getting_started/{3_0_new_embodiment_finetuning,4_deeper_understanding,5_policy_deployment}.md`.
Also NVIDIA's SO-101 tutorial (huggingface.co/blog/nvidia/gr00t-n1-5-so101-tuning).

- Loading: `Gr00tPolicy(model_path, embodiment_tag, modality_config,
  modality_transform, denoising_steps, device)` (README 2.1). Its own
  `snapshot_download(model_path)` and `GR00T_N1_5.from_pretrained()` take NO
  revision (they resolve `main`). So `_download_pinned` fetches the reviewed
  revision into a local dir, and the local path is passed instead. A local path
  is not a valid repo id, so both calls fall back to it (`HFValidationError`).
  The Eagle 2.5 VLM config and tokenizer are bundled in the pinned clone
  (`eagle2_hg_model/`); every weight comes from the checkpoint's safetensors, so
  the checkpoint revision is the only network fetch.
- Embodiment: SO-100/SO-101 were NOT in N1.5's pre-training. The tutorial
  fine-tunes them as `new_embodiment` with data config `so100_dualcam`
  (`video.front` + `video.wrist`, `state/action.single_arm` [5] +
  `.gripper` [1], `annotation.human.task_description`, 1 observation step,
  16-step action chunk). The base `nvidia/GR00T-N1.5-3B`'s
  `experiment_cfg/metadata.json` has only `gr1`, `oxe_droid` and
  `agibot_genie1`, so `Gr00tPolicy(..., "new_embodiment")` on the base raises
  "No metadata found for embodiment tag". A real SO-101 fine-tune is required.
- Action convention: `so100_dualcam` normalizes `single_arm`/`gripper` with
  `min_max` (`StateActionTransform`); `get_action()` un-applies it, so the chunk
  comes back in the dataset's raw units. Every SO-101 N1.5 fine-tune's metadata
  marks both action groups `absolute: true`. NVIDIA's own real-robot client
  (`examples/SO-100/eval_lerobot.py`) feeds `robot.get_observation()` values
  straight in as `state.*`, and sends each chunk step straight to
  `robot.send_action()`. So the actions are ABSOLUTE lerobot `.pos` joint
  targets, with no delta and no IK, in whatever unit mode the dataset was
  recorded in. That mode is checked per checkpoint (`classify_action_units`).
- Units: lerobot SO-101 data comes in several modes. Current lerobot
  (`so_follower`, `use_degrees=True`, the mode `action_contract.py` speaks) is
  degrees about the calibrated midpoint. Older SO-100 data is range-normalized
  (+/-100, clamped at the calibration limits), or uses the pre-0.2 offset-zero
  calibration (shoulder_lift up to ~199). Only degree mode is an identity here,
  so the reviewed profile pins it and `_check_checkpoint_layout` re-derives it
  from the downloaded `metadata.json` before the model loads.
- Images: `Gr00tPolicy` puts the transforms in eval mode: `VideoCrop` becomes a
  0.95 centre crop and `VideoColorJitter` is skipped. `VideoCrop.check_input`
  asserts the frame is exactly the training resolution from metadata (640x480),
  so frames are resized to it here. Colors are RGB (lerobot `OpenCVCamera`).
- flash-attn: the bundled Eagle config hard-codes `flash_attention_2` and
  asserts it for the Qwen3 LLM, so the GPU must be Ampere or newer (A100/L4). A
  T4 (sm75) cannot run it. `main()` refuses such GPUs.
- Pins (pyproject `[base]` + README): Python 3.10, `torch==2.5.1`,
  `transformers==4.51.3`, `flash-attn==2.7.1.post4`. `main()` refuses others.

Checkpoint (HF API, read 2026-10-01; see `CHECKPOINT_PROFILES`)
---------------------------------------------------------------
`quangnd58/GR00T-N1.5-3B-so101-multitask` @ 488f3b6. Model card: base
`nvidia/GR00T-N1.5-3B`, trained with Isaac-GR00T branch `n1d5`
(`scripts/gr00t_finetune.py`), `new_embodiment`, `so100_dualcam`,
`video.front` = top camera, `video.wrist`, 640x480, horizon 16, license
"other" (NVIDIA License, derived from the base). Its 899 weight keys equal the
official base's. Dataset `hungho77/so101-multitask` (Apache-2.0): lerobot v3.0,
`robot_type: so_follower`, 30 fps, 143 episodes, 3 tasks, names exactly
`shoulder_pan.pos .. gripper.pos`. Its stats are degree mode (wrist_flex
reaches 100.4, which is past the range mode's +/-100 clamp; no joint sits at
exactly +/-100). Task prompts (use verbatim): "Pick up the banana and place it
in the bot, then close the lid", "Pick blue cube and place on red cube", "Pick
all cubes and place into cup".

Known limitations (documented, not hidden)
------------------------------------------
- The default checkpoint is a small community fine-tune (16 downloads, published
  2026-09-29) on someone else's rig and scene. Expect pipeline-proving motion,
  not task success.
- `video.front` is the dataset's TOP camera, mapped to this rig's
  `overhead_left` by default (`--front-image-key` overrides).
- It returns one chunk step about one client control period ahead (14 at the
  default 2 Hz), clamped to the 16-step horizon, not NVIDIA's "execute 8 steps
  at 50 Hz" client loop. Every step is an absolute target, so the per-step
  safety clamp bounds the jump.

Colab launch: see vla_bridge/notebooks/http_groot_n15.ipynb.
"""
import argparse
import base64
import json
import math
import numbers
import subprocess
import threading
import time
from pathlib import Path

import cv2
import numpy as np

DEFAULT_CHECKPOINT = "quangnd58/GR00T-N1.5-3B-so101-multitask"
DEFAULT_LOCAL_DIR = "/content/groot_n15_checkpoints"
DEFAULT_GROOT_REPO = "/content/Isaac-GR00T"
DEFAULT_PORT = 3006  # 3000 FastFS, 3001 OpenVLA-OFT, 3002 Wall-X, 3003 OpenVLA, 3004 Octo, 3005 RDT-1B
DEFAULT_CLIENT_CONTROL_HZ = 2.0  # run_vla_episode.py's default --control-hz
DEFAULT_INSTRUCTION = "Pick blue cube and place on red cube"
# github.com/NVIDIA/Isaac-GR00T tag n1.5-release == branch n1d5 (GitHub API, 2026-10-01).
ISAAC_GROOT_COMMIT = "4af2b622892f7dcb5aae5a3fb70bcb02dc217b96"

# Reviewed checkpoints. `backend_configs/groot_n15.json["checkpoints"]` must agree
# (pinned by test_client_and_server_checkpoint_profiles_agree).
CHECKPOINT_PROFILES = {
    "quangnd58/GR00T-N1.5-3B-so101-multitask": dict(
        revision="488f3b6da322e55a316a562a00567fb4a1fe130a", subfolder="", data_config="so100_dualcam",
        embodiment_tag="new_embodiment", video_sources={"front": "overhead_left", "wrist": "wrist"},
        action_space="absolute_joint", action_units="degrees", control_hz=30, action_horizon=16,
        denoising_steps=4),
}

# 6 DOF in action_contract.JOINT_ORDER (re-declared: this file runs standalone on Colab).
SO101_JOINTS = ("shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll", "gripper")
ARM_GROUP, GRIPPER_GROUP = "single_arm", "gripper"
GROUP_SHAPES = {ARM_GROUP: 5, GRIPPER_GROUP: 1}

MAX_IMAGE_SIDE = 1024
MAX_INSTRUCTION_CHARS = 1000
IMAGE_KEYS = ("overhead_left", "overhead_right", "wrist")
MIN_COMPUTE_CAPABILITY = (8, 0)  # flash-attn 2: Ampere or newer
PINNED_DEPENDENCIES = {"torch": {"2.5.1"}, "transformers": {"4.51.3"}, "flash_attn": {"2.7.1.post4"}}


def chunk_step_for(control_hz, client_hz, horizon):
    """Chunk step i is the absolute target i/control_hz s ahead; return the one
    about one client control period ahead, clamped to the chunk."""
    step = int(round(float(control_hz) / float(client_hz))) - 1
    return max(0, min(int(horizon) - 1, step))


def classify_action_units(action_stats):
    """Unit mode of an SO-101 checkpoint, from its own `single_arm` action stats.

    - `radians`: every bound within +/-3.5.
    - `legacy_offset`: some bound past 180 (the pre-0.2 lerobot SO-100 calibration).
    - `degrees`: some bound past 100, which the range mode's clamp cannot produce.
    - `range_m100_100`: a bound at exactly +/-100 (lerobot clamps there), none past it.
    - `ambiguous`: anything else. It is not accepted as degrees.
    """
    arm = action_stats.get(ARM_GROUP) or {}
    values = [abs(float(v)) for v in list(arm.get("min", [])) + list(arm.get("max", []))]
    if not values:
        return "ambiguous"
    peak = max(values)
    if peak <= 3.5:
        return "radians"
    if peak > 180.0:
        return "legacy_offset"
    if peak > 100.0 + 1e-6:
        return "degrees"
    if any(abs(v - 100.0) <= 1e-4 for v in values):
        return "range_m100_100"
    return "ambiguous"


def _download_pinned(checkpoint, revision, local_dir, subfolder):
    """Materialise the pinned revision's inference files (no optimizer state) as a
    real directory; return the model directory (`<snapshot>/<subfolder>`)."""
    from huggingface_hub import snapshot_download

    target = Path(local_dir) / checkpoint.replace("/", "--") / revision
    patterns = [f"{subfolder}{name}" for name in (
        "config.json", "experiment_cfg/*", "model*.safetensors", "model.safetensors.index.json")]
    snapshot_download(repo_id=checkpoint, revision=revision, local_dir=str(target), allow_patterns=patterns)
    return str(target / subfolder) if subfolder else str(target)


def _git_head(repo):
    result = subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True, text=True)
    return result.stdout.strip() if result.returncode == 0 else None


def _is_groot_checkout(repo):
    return (Path(repo) / "gr00t" / "model" / "policy.py").is_file()


def _check_groot_checkout(groot_repo):
    """Refuse to start unless `groot_repo` is the pinned Isaac-GR00T commit (T-14-06)."""
    if not _is_groot_checkout(groot_repo):
        raise SystemExit(f"{groot_repo} is not an Isaac-GR00T checkout (no gr00t/model/policy.py)")
    head = _git_head(groot_repo)
    if head != ISAAC_GROOT_COMMIT:
        raise SystemExit(f"{groot_repo} is at {head}, expected the reviewed Isaac-GR00T n1.5-release commit "
                         f"{ISAAC_GROOT_COMMIT}")


def _read_checkpoint_metadata(model_dir, embodiment_tag):
    path = Path(model_dir) / "experiment_cfg" / "metadata.json"
    metadata = json.loads(path.read_text())
    if embodiment_tag not in metadata:
        raise RuntimeError(f"{path} has no {embodiment_tag!r} entry (found {sorted(metadata)}); "
                           "Gr00tPolicy would refuse it")
    return metadata[embodiment_tag]


def _check_checkpoint_layout(metadata, profile):
    """The downloaded checkpoint must be the layout and unit mode its profile was
    reviewed as. Raises `RuntimeError` before the 3B model is loaded."""
    modalities = metadata.get("modalities", {})
    videos = sorted(modalities.get("video", {}))
    if videos != sorted(profile["video_sources"]):
        raise RuntimeError(f"checkpoint video keys {videos} != reviewed {sorted(profile['video_sources'])}")
    for modality in ("state", "action"):
        groups = modalities.get(modality, {})
        if sorted(groups) != sorted(GROUP_SHAPES):
            raise RuntimeError(f"checkpoint {modality} groups {sorted(groups)} != {sorted(GROUP_SHAPES)}")
        for group, size in GROUP_SHAPES.items():
            if list(groups[group].get("shape", [])) != [size]:
                raise RuntimeError(f"checkpoint {modality}.{group} shape {groups[group].get('shape')} != [{size}]")
            if groups[group].get("absolute") is not True:
                raise RuntimeError(f"checkpoint {modality}.{group} is not absolute; this adapter only "
                                   "serves absolute joint targets (Pitfall 3)")
    detected = classify_action_units(metadata.get("statistics", {}).get("action", {}))
    if detected != profile["action_units"]:
        raise RuntimeError(f"checkpoint action stats look like {detected!r}, but its profile says "
                           f"{profile['action_units']!r}; refusing to serve it as an identity mapping")
    return detected


def _load_policy(model_dir, profile, denoising_steps, groot_repo):
    """README 2.1: `Gr00tPolicy` with the profile's data config, from a LOCAL path."""
    import sys

    root = str(Path(groot_repo).resolve())
    if not _is_groot_checkout(root):
        raise RuntimeError(f"{root} is not an Isaac-GR00T checkout")
    if root not in sys.path:
        sys.path.insert(0, root)
    import gr00t

    if not str(Path(gr00t.__file__).resolve()).startswith(root):
        raise RuntimeError(f"imported gr00t from {gr00t.__file__}, not the pinned checkout at {root}")
    from gr00t.experiment.data_config import DATA_CONFIG_MAP
    from gr00t.model.policy import Gr00tPolicy

    data_config = DATA_CONFIG_MAP[profile["data_config"]]
    policy = Gr00tPolicy(
        model_path=str(model_dir),
        embodiment_tag=profile["embodiment_tag"],
        modality_config=data_config.modality_config(),
        modality_transform=data_config.transform(),
        denoising_steps=denoising_steps,
        device="cuda",
    )
    return policy, data_config


def _verify_weights(model, model_dir):
    """`GR00T_N1_5.from_pretrained` (transformers) only warns on mismatched keys.
    Fail instead: every model key must come from the checkpoint, or be tied to one
    that does (Qwen's lm_head), and the checkpoint must hold no unused key."""
    index = json.loads((Path(model_dir) / "model.safetensors.index.json").read_text())
    saved = set(index["weight_map"])
    state = model.state_dict()
    unexpected = sorted(saved - set(state))
    loaded_ptrs = {state[k].data_ptr() for k in saved & set(state)}
    missing = sorted(k for k in set(state) - saved if state[k].data_ptr() not in loaded_ptrs)
    if unexpected or missing:
        raise RuntimeError(f"checkpoint weights do not match the model: missing {missing[:5]} "
                           f"({len(missing)}), unexpected {unexpected[:5]} ({len(unexpected)})")


def _get_action(policy, observation):
    return policy.get_action(observation)


def parse_joint_state(joint_state):
    """`{JOINT_ORDER name: finite number}` -> floats in `SO101_JOINTS` order."""
    if not isinstance(joint_state, dict):
        raise TypeError("joint_state must be an object")
    values = []
    for joint in SO101_JOINTS:
        value = joint_state[joint]
        if isinstance(value, bool) or not isinstance(value, numbers.Real):
            raise TypeError(f"joint_state[{joint!r}] must be a number")
        if not math.isfinite(value):
            raise ValueError(f"joint_state[{joint!r}] must be finite")
        values.append(float(value))
    return values


class GrootN15Model:
    def __init__(self, checkpoint=DEFAULT_CHECKPOINT, revision=None, groot_repo=DEFAULT_GROOT_REPO,
                 local_dir=DEFAULT_LOCAL_DIR, client_control_hz=DEFAULT_CLIENT_CONTROL_HZ,
                 front_image_key=None, denoising_steps=None):
        profile = CHECKPOINT_PROFILES.get(checkpoint)
        if profile is None:
            raise ValueError(f"{checkpoint} has no reviewed profile (known: {sorted(CHECKPOINT_PROFILES)})")
        revision = revision or profile["revision"]
        video_sources = dict(profile["video_sources"])
        if front_image_key is not None:
            if front_image_key not in IMAGE_KEYS:
                raise ValueError(f"front_image_key must be one of {IMAGE_KEYS}")
            video_sources["front"] = front_image_key
        denoising_steps = int(denoising_steps or profile["denoising_steps"])

        model_dir = _download_pinned(checkpoint, revision, local_dir, profile["subfolder"])
        checkpoint_metadata = _read_checkpoint_metadata(model_dir, profile["embodiment_tag"])
        detected_units = _check_checkpoint_layout(checkpoint_metadata, profile)
        self.policy, data_config = _load_policy(model_dir, profile, denoising_steps, groot_repo)
        _verify_weights(self.policy.model, model_dir)

        self.video_keys = {key: key.split(".", 1)[1] for key in data_config.video_keys}
        if sorted(self.video_keys.values()) != sorted(video_sources):
            raise RuntimeError(f"data config {profile['data_config']} video keys {data_config.video_keys} do not "
                               f"match the profile's {sorted(video_sources)}")
        self.state_keys = list(data_config.state_keys)
        self.action_keys = list(data_config.action_keys)
        if self.state_keys != [f"state.{g}" for g in GROUP_SHAPES] or \
                self.action_keys != [f"action.{g}" for g in GROUP_SHAPES]:
            raise RuntimeError(f"data config {profile['data_config']} is not single_arm + gripper")
        self.language_key = data_config.language_keys[0]
        self.action_horizon = len(data_config.action_indices)
        self.video_sources = video_sources
        video_meta = checkpoint_metadata["modalities"]["video"]
        self.resolutions = {name: tuple(int(v) for v in video_meta[name]["resolution"])  # (width, height)
                            for name in video_sources}
        width, height = self.resolutions[next(iter(self.resolutions))]
        self.warmup_shape = (height, width, 3)
        self.required_image_keys = tuple(dict.fromkeys(video_sources.values()))
        self.chunk_step = chunk_step_for(profile["control_hz"], client_control_hz, self.action_horizon)
        self.model_version = f"{checkpoint}@{revision[:12]}"
        self.metadata = dict(
            checkpoint=checkpoint, revision=revision, isaac_groot_commit=ISAAC_GROOT_COMMIT,
            data_config=profile["data_config"], embodiment_tag=profile["embodiment_tag"],
            action_space=profile["action_space"], action_units=profile["action_units"],
            detected_action_units=detected_units, action_horizon=self.action_horizon,
            control_hz=profile["control_hz"], client_control_hz=float(client_control_hz),
            chunk_step=self.chunk_step, denoising_steps=denoising_steps, video_sources=video_sources,
            video_resolution={k: list(v) for k, v in self.resolutions.items()},
            joint_order=list(SO101_JOINTS),
            returns="one absolute joint-target step of the un-normalized action chunk, JOINT_ORDER",
        )

    def _frame(self, name, rgb):
        width, height = self.resolutions[name]
        if rgb.shape[:2] != (height, width):
            rgb = cv2.resize(rgb, (width, height), interpolation=cv2.INTER_AREA)
        return np.ascontiguousarray(rgb, dtype=np.uint8)[None]

    def __call__(self, images, instruction, joint_state):
        state = parse_joint_state(joint_state)
        observation = {key: self._frame(name, images[self.video_sources[name]])
                       for key, name in self.video_keys.items()}
        observation[f"state.{ARM_GROUP}"] = np.asarray([state[:5]], dtype=np.float64)
        observation[f"state.{GRIPPER_GROUP}"] = np.asarray([state[5:]], dtype=np.float64)
        observation[self.language_key] = [instruction]
        chunk = _get_action(self.policy, observation)
        try:
            arm = np.asarray(chunk[f"action.{ARM_GROUP}"], dtype=np.float64)
            gripper = np.asarray(chunk[f"action.{GRIPPER_GROUP}"], dtype=np.float64)
        except (KeyError, TypeError, ValueError) as exc:
            raise RuntimeError(f"model returned an unexpected action chunk ({exc!r})") from exc
        if gripper.ndim == 1:
            gripper = gripper[:, None]
        if arm.shape != (self.action_horizon, 5) or gripper.shape != (self.action_horizon, 1):
            raise RuntimeError(f"model returned single_arm {arm.shape} / gripper {gripper.shape}, expected "
                               f"({self.action_horizon}, 5) / ({self.action_horizon}, 1)")
        values = np.concatenate([arm[self.chunk_step], gripper[self.chunk_step]])
        if not np.all(np.isfinite(values)):
            raise RuntimeError(f"model returned a non-finite action {values.tolist()}")
        return [float(v) for v in values]


def _decode_rgb(b64_png):
    """base64 PNG (OpenCV BGR-encoded by the client) -> HxWx3 uint8 RGB."""
    if not isinstance(b64_png, str):
        raise TypeError("image must be a base64 string")
    raw = base64.b64decode(b64_png, validate=True)
    frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError("invalid PNG")
    if max(frame.shape[:2]) > MAX_IMAGE_SIDE:
        raise ValueError(f"image dimensions must be <= {MAX_IMAGE_SIDE}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def create_app(infer):
    from flask import Flask, jsonify, request
    app = Flask(__name__)
    app.config['MAX_CONTENT_LENGTH'] = 20 * 1024 * 1024
    inference_lock = threading.Lock()
    image_keys = tuple(getattr(infer, 'required_image_keys', ('overhead_left', 'wrist')))

    @app.get('/health')
    def health():
        return jsonify(status='ready', protocol_version=1, model=infer.metadata)

    @app.post('/action')
    def action():
        if not inference_lock.acquire(blocking=False):
            return jsonify(error='groot n1.5 service busy'), 429
        try:
            body = request.get_json(silent=True)
            if not isinstance(body, dict):
                raise ValueError('expected a JSON object body')
            images_in = body['images']
            if not isinstance(images_in, dict):
                raise TypeError('images must be an object')
            images = {key: _decode_rgb(images_in[key]) for key in image_keys}
            instruction = body['instruction']
            if not isinstance(instruction, str) or not instruction.strip():
                raise ValueError('instruction must be a non-empty string')
            if len(instruction) > MAX_INSTRUCTION_CHARS:
                raise ValueError(f'instruction must be <= {MAX_INSTRUCTION_CHARS} characters')
            joint_state = body['joint_state']
            parse_joint_state(joint_state)  # GR00T needs the proprio: reject before the model runs
            reset = body.get('reset', False)
            if not isinstance(reset, bool):
                raise TypeError('reset must be a boolean')  # accepted for wire parity; GR00T is stateless
            started = time.perf_counter()
            result = infer(images, instruction, joint_state)
            elapsed = (time.perf_counter() - started) * 1000
            return jsonify(action=[float(v) for v in result], model_version=infer.model_version,
                           inference_ms=elapsed)
        except (ValueError, KeyError, TypeError) as exc:
            return jsonify(error=str(exc)), 400
        except RuntimeError as exc:
            return jsonify(error=f'groot n1.5 inference failed: {exc}'), 500
        finally:
            inference_lock.release()
    return app


def _installed_versions():
    import importlib.metadata as metadata

    return {name: metadata.version(name).split("+")[0] for name in PINNED_DEPENDENCIES}


def _check_dependency_versions():
    versions = _installed_versions()
    bad = {name: version for name, version in versions.items() if version not in PINNED_DEPENDENCIES[name]}
    if bad:
        raise SystemExit(
            "GR00T N1.5 needs torch==2.5.1, transformers==4.51.3, flash-attn==2.7.1.post4 "
            f"(Isaac-GR00T n1.5-release pyproject/README); found {bad}. Use the isolated GR00T N1.5 venv."
        )


def _gpu_info():
    import torch

    if not torch.cuda.is_available():
        return dict(cuda=False, name=None, total_gib=0.0, capability=None, bf16=False)
    props = torch.cuda.get_device_properties(0)
    return dict(cuda=True, name=props.name, total_gib=props.total_memory / 2**30,
                capability=tuple(torch.cuda.get_device_capability(0)), bf16=bool(torch.cuda.is_bf16_supported()))


def _peak_vram_gib():
    import torch

    return torch.cuda.max_memory_allocated() / 2**30


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--checkpoint', default=DEFAULT_CHECKPOINT, choices=sorted(CHECKPOINT_PROFILES))
    parser.add_argument('--revision', default=None, help="pinned HF commit sha (defaults to the reviewed profile's)")
    parser.add_argument('--groot-repo', default=DEFAULT_GROOT_REPO,
                        help=f'NVIDIA/Isaac-GR00T checkout at {ISAAC_GROOT_COMMIT[:7]} (n1.5-release)')
    parser.add_argument('--local-dir', default=DEFAULT_LOCAL_DIR)
    parser.add_argument('--front-image-key', default=None, choices=IMAGE_KEYS,
                        help="rig camera fed as video.front (default: the profile's, overhead_left)")
    parser.add_argument('--denoising-steps', type=int, default=None, help="default: the profile's (4)")
    parser.add_argument('--client-control-hz', type=float, default=DEFAULT_CLIENT_CONTROL_HZ,
                        help="run_vla_episode.py's --control-hz; picks the returned chunk step")
    parser.add_argument('--warmup-instruction', default=DEFAULT_INSTRUCTION)
    parser.add_argument('--port', type=int, default=DEFAULT_PORT)
    args = parser.parse_args()
    _check_dependency_versions()
    _check_groot_checkout(args.groot_repo)
    gpu = _gpu_info()
    if not gpu['cuda'] or gpu['capability'] is None or tuple(gpu['capability']) < MIN_COMPUTE_CAPABILITY \
            or not gpu['bf16']:
        raise SystemExit(f"GR00T N1.5's Eagle backbone requires flash_attention_2, which needs an Ampere+ GPU "
                         f"with bf16 (A100/L4); found {gpu}. A T4 cannot run it.")
    print(f"GPU {gpu['name']} {gpu['total_gib']:.1f} GiB, sm{gpu['capability'][0]}{gpu['capability'][1]}",
          flush=True)
    model = GrootN15Model(args.checkpoint, revision=args.revision, groot_repo=args.groot_repo,
                          local_dir=args.local_dir, client_control_hz=args.client_control_hz,
                          front_image_key=args.front_image_key, denoising_steps=args.denoising_steps)
    # First inference compiles kernels / allocates; it happens before readiness is reported.
    model({key: np.zeros(model.warmup_shape, dtype=np.uint8) for key in model.required_image_keys},
          args.warmup_instruction, {joint: 0.0 for joint in SO101_JOINTS})
    print(f'GR00T N1.5 model loaded and warmed ({model.model_version}); peak VRAM {_peak_vram_gib():.1f} GiB; ready',
          flush=True)
    create_app(model).run(host='0.0.0.0', port=args.port, threaded=True, use_reloader=False)


if __name__ == '__main__':
    main()


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '23dcd2dc50c9ce593d7ab95ff2ad92c43253c46eb6e5463aabdcfe40d4b1a54e'
actual = hashlib.sha256(Path('/content/groot_n15_server.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/groot_n15_server.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/groot_n15_server.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/groot_n15_server.py matches control/vla_bridge/groot_n15_server.py (sha256 {actual[:12]})')

## Launch

The launch cell first stops any earlier server: every process whose command line contains one of `groot_n15_server.py`, and whatever listens on port 3006.
It then starts this notebook's server as a separate OS process (unbuffered, log at `/content/groot_n15_server.log`).
It only succeeds once **its own** still-running process owns port 3006.
A stale server answering on the port is an error, never a successful launch (in the 2026-10-18 live session a stale stock server made a diffusion launch look successful).

In [ ]:
import shutil
import sys

sys.path.insert(0, '/content')
from colab_server_utils import start_server, stop_servers, wait_until_ready

PORT = 3006
GROOT_N15_LOG = '/content/groot_n15_server.log'
STOP_PATTERNS = ['groot_n15_server.py']
GROOT_N15_DIR = '/content/Isaac-GR00T'
GROOT_N15_PYTHON = '/content/groot_n15_venv/bin/python'

free_gb = shutil.disk_usage('/content').free / 1e9
if free_gb < 12:
    raise RuntimeError(f'only {free_gb:.0f} GB free on /content; GR00T N1.5 needs ~10 GB (7.6 GB checkpoint)')

stop_servers(PORT, STOP_PATTERNS)
groot_n15_server_process = start_server(
    [GROOT_N15_PYTHON, '-u', '/content/groot_n15_server.py', '--port', '3006', '--groot-repo', GROOT_N15_DIR],
    GROOT_N15_LOG,
    env_overrides={'TOKENIZERS_PARALLELISM': 'false'},
    cwd='/content',
)
health = wait_until_ready(
    groot_n15_server_process, PORT, GROOT_N15_LOG,
    timeout_s=3600, health_url='http://127.0.0.1:3006/health',
)
print('GR00T N1.5 adapter ready:', {k: health['model'][k] for k in (
    'checkpoint', 'revision', 'isaac_groot_commit', 'action_space', 'action_units',
    'detected_action_units', 'chunk_step', 'video_sources')})
print([line for line in open(GROOT_N15_LOG) if 'GiB' in line or 'OutOfMemory' in line])

## Open an HTTP tunnel to port 3006

In [ ]:
from pyngrok import ngrok

groot_n15_tunnel = ngrok.connect(3006, "http")  # plain JSON-over-HTTP
print(f"GR00T N1.5 endpoint: {groot_n15_tunnel.public_url}")
SMOKE = ".venv/bin/python -m vla_bridge.backend_smoke --backend groot-n15 --server-address https://<groot_n15_tunnel> --instruction 'Pick blue cube and place on red cube'"
EPISODE = '.venv/bin/python run_vla_episode.py --backend groot-n15 --server-address https://GROOT_N15_TUNNEL.ngrok-free.app --instruction "Pick blue cube and place on red cube" --out outputs/groot_n15_UNIQUE_NAME --max-steps 20'
for placeholder in ('https://<groot_n15_tunnel>', 'https://GROOT_N15_TUNNEL.ngrok-free.app'):
    SMOKE = SMOKE.replace(placeholder, groot_n15_tunnel.public_url)
    EPISODE = EPISODE.replace(placeholder, groot_n15_tunnel.public_url)
print('Local smoke (run in control/):')
print(SMOKE)
print('Local episode (run in control/):')
print(EPISODE)

## Run locally

Then, locally in `control/` (pass the base URL; the client appends `/action`):

```bash
.venv/bin/python run_vla_episode.py --backend groot-n15 \
  --server-address https://GROOT_N15_TUNNEL.ngrok-free.app \
  --instruction "Pick blue cube and place on red cube" \
  --out outputs/groot_n15_UNIQUE_NAME --max-steps 20
```

Each step's `model_version` should read `quangnd58/GR00T-N1.5-3B-so101-multitask@488f3b6da322`. The action
is an absolute joint target in degrees (gripper in percent), passed straight through: no unit change, no
IK, and the current joint state is never added to it. Every action then goes through the safety validator.
The checkpoint was trained on three prompts; use one verbatim: "Pick up the banana and place it in the bot,
then close the lid", "Pick blue cube and place on red cube", "Pick all cubes and place into cup".

A step that logs `http-error-holding-position` means the adapter returned an error. Check
`/content/groot_n15_server.log`:
- 500 `groot n1.5 inference failed` means the model produced a non-finite or wrong-shaped action chunk.
- 400 means a bad request, e.g. a missing camera or joint.

What a units mistake would look like (it is checked at startup, so this should not happen): a joint driven
toward its limit on the very first step, every step clamped by the per-step limit in the same direction.
If you see that, stop and compare `/health`'s `detected_action_units` with the checkpoint's metadata.
Expect motion that is not task-meaningful: the checkpoint was trained on another rig and scene.

Smoke run first (synthetic hardware, the arm cannot move; the tunnel cell prints it with the URL filled in):

```bash
.venv/bin/python -m vla_bridge.backend_smoke --backend groot-n15 --server-address https://<groot_n15_tunnel> --instruction 'Pick blue cube and place on red cube'
```

## Diagnostics

Run any of these after the launch cell, also from a restarted kernel: each cell re-declares
what it needs.
- Log: the tail of `/content/groot_n15_server.log`, then every line matching `ERROR`, `Error`, `Traceback`,
  `Exception`, `OutOfMemory`, `out of memory`, `Killed` or `beginAllocateToPool`.
- Processes: every known server process on this runtime (stale servers from other notebooks
  show up here); fails unless `groot_n15_server.py` owns port 3006.
- Package versions, GPU memory, and `/health`.

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import log_report

log_report('/content/groot_n15_server.log')

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import process_report

process_report(3006, 'groot_n15_server.py', globals().get('groot_n15_server_process'))

In [ ]:
import subprocess

GROOT_N15_PYTHON = '/content/groot_n15_venv/bin/python'
probe = (
    "import importlib.metadata as m\n"
    "for p in ('torch', 'transformers', 'flash_attn', 'numpy', 'huggingface_hub'):\n"
    "    try:\n"
    "        print(p, m.version(p))\n"
    "    except m.PackageNotFoundError:\n"
    "        print(p, 'not installed')\n"
)
result = subprocess.run([GROOT_N15_PYTHON, '-c', probe], capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-2000:])

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import gpu_report

gpu_report()

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import health_report

health_report(3006)

## Teardown (every session, no exceptions)

An unattended open tunnel into a Colab-hosted server is an
elevation-of-privilege risk (T-11-08 in this plan's threat register) —
stop both processes at the end of every session, not just when finished
for the day: the cell below disconnects this notebook's tunnel, stops the
ngrok agent, stops the server and confirms port 3006 is free. Run it only
after the local episode or smoke run has finished.

In [ ]:
# GR00T N1.5 teardown -- every session, after the local run has finished.
import sys

sys.path.insert(0, '/content')
from colab_server_utils import stop_server
from pyngrok import ngrok

if 'groot_n15_tunnel' in globals():
    try:
        ngrok.disconnect(groot_n15_tunnel.public_url)
        print(f'Disconnected {groot_n15_tunnel.public_url}')
    except Exception as exc:
        print(f'Tunnel already closed ({type(exc).__name__}: {exc})')
ngrok.kill()
print('ngrok agent stopped.')
stop_server(globals().get('groot_n15_server_process'), 3006, ['groot_n15_server.py'])